# YOLO11-seg — açaí e triatomíneos (versão revisada para o artigo)

Este notebook refaz os experimentos do artigo atendendo ao **item 7 do parecer técnico**:

| Pedido do parecer | Onde é atendido |
|---|---|
| Mais épocas e base maior | Seção 2 (várias fontes de imagens) e Seção 3 (`EPOCHS=300`, `patience=50`) |
| Separar dados por origem; impedir que imagens aumentadas atravessem partições | Seção 4: divisão **por grupo** (foto de origem / vídeo / fonte) + verificação automática de vazamento |
| Teste externo e independente | Seção 4: `HOLDOUT_SOURCES` (ex.: fotos de campo novas vão inteiras para o teste) |
| Repetições com sementes diferentes; médias e IC | Seções 5 e 6 (`SEEDS`), média ± DP e IC 95% (t de Student) |
| Resultados por classe, matriz de confusão, curvas P-R | Seções 5–7 (salvos em `figuras/` e `tabelas/`) |
| Falsos negativos de triatomíneos | Seção 5: auditoria por limiar de confiança + lista de imagens perdidas |
| Registrar versões, pesos, sementes, critério de seleção | `run_metadata.json` |

**Como usar:** `Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)` e rodar as células em ordem.
Tempo estimado na T4: ~3–6 min por época-1000-imagens com `yolo11s-seg`; 5 sementes × 300 épocas com early stopping costuma levar 1–3 h para ~300 imagens.


## 1. Ambiente

In [ ]:
!nvidia-smi -L || echo "SEM GPU: ative em Ambiente de execução > Alterar o tipo de ambiente"
%pip install -q "ultralytics==8.4.182" roboflow pandas matplotlib pyyaml
import ultralytics, torch; print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
%%writefile acai_yolo_pipeline.py
"""
Pipeline de treino e avaliação YOLO11-seg para o artigo do açaí (prova de conceito).

Responde aos itens do parecer técnico (seção 7):
  - re-divisão treino/validação/teste por GRUPO (imagem de origem), impedindo que
    variantes aumentadas da mesma foto atravessem partições;
  - aumento de dados apenas no treino (on-the-fly, pelo Ultralytics);
  - várias sementes, com média, desvio-padrão e IC 95%;
  - resultados no conjunto de TESTE, por classe (caixa e máscara);
  - matriz de confusão e curvas P-R (salvas pelo Ultralytics);
  - análise explícita de falsos negativos de triatomíneos;
  - registro de versões, pesos iniciais, sementes e critério de seleção.

Uso (Colab):  ver o notebook YOLO11_acai_triatomineos.ipynb
"""
from __future__ import annotations

import json
import math
import platform
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import yaml

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# ----------------------------------------------------------------------------
# 1. Coleta de imagens + rótulos de um ou mais datasets no formato YOLO
# ----------------------------------------------------------------------------
def group_key(stem: str) -> str:
    """Identifica a imagem de ORIGEM.

    O Roboflow nomeia arquivos como  IMG_1234_jpg.rf.<hash>.jpg ; versões
    aumentadas da mesma foto compartilham o prefixo antes de '.rf.'.
    Também remove sufixos comuns de frames de vídeo (ex.: video01_frame_0042),
    para que frames do mesmo vídeo fiquem no mesmo grupo.
    """
    base = stem.split(".rf.")[0]
    base = re.sub(r"_(jpg|jpeg|png)$", "", base, flags=re.I)
    base = re.sub(r"[_-]?(frame|f)[_-]?\d+$", "", base, flags=re.I)
    return base.lower()


def collect_items(dataset_dirs: list[str], source_tags: list[str] | None = None):
    """Varre pastas de datasets YOLO (com subpastas train/valid/test ou não)."""
    items = []
    for i, d in enumerate(dataset_dirs):
        d = Path(d)
        tag = source_tags[i] if source_tags else d.name
        for img in d.rglob("*"):
            if img.suffix.lower() not in IMG_EXT or "images" not in img.parts:
                continue
            parts = list(img.parts)
            idx = len(parts) - 1 - parts[::-1].index("images")
            parts[idx] = "labels"
            lbl = Path(*parts).with_suffix(".txt")
            classes = []
            if lbl.exists():
                for line in lbl.read_text().splitlines():
                    if line.strip():
                        classes.append(int(float(line.split()[0])))
            items.append({
                "img": str(img),
                "lbl": str(lbl) if lbl.exists() else None,
                "group": f"{tag}::{group_key(img.stem)}",
                "source": tag,
                "classes": classes,
            })
    return items


def dedupe_augmented(items, keep="first"):
    """Mantém uma única imagem por grupo (descarta variantes aumentadas
    pelo Roboflow). Recomendado: o aumento passa a ser feito só no treino."""
    seen, out = set(), []
    for it in sorted(items, key=lambda x: x["img"]):
        if it["group"] in seen:
            continue
        seen.add(it["group"])
        out.append(it)
    return out


# ----------------------------------------------------------------------------
# 2. Divisão por grupo, estratificada pela presença de classes
# ----------------------------------------------------------------------------
def group_split(items, fractions=(0.70, 0.15, 0.15), seed=0, holdout_sources=None, train_only_patterns=("gemini",)):
    """Divide por grupo. Se `holdout_sources` for dado (ex.: ['campo_2026']),
    essas fontes vão inteiras para o TESTE EXTERNO e o restante é dividido
    entre treino e validação."""
    rng = random.Random(seed)
    groups = defaultdict(list)
    for it in items:
        groups[it["group"]].append(it)

    split = {"train": [], "val": [], "test": []}
    pool = []
    for g, its in groups.items():
        # imagens sintéticas (geradas por IA) entram só no treino, nunca na validação/teste
        if train_only_patterns and any(pat in g.lower() for pat in train_only_patterns):
            split["train"].extend(its)
        elif holdout_sources and its[0]["source"] in holdout_sources:
            split["test"].extend(its)
        else:
            pool.append(g)

    # estrato = conjunto de classes presentes no grupo (ex.: (0,), (1,), (0,1), ())
    strata = defaultdict(list)
    for g in pool:
        cls = tuple(sorted({c for it in groups[g] for c in it["classes"]}))
        strata[cls].append(g)

    ftr, fva, fte = fractions
    if holdout_sources:
        ftr, fva, fte = ftr / (ftr + fva), fva / (ftr + fva), 0.0
    for cls, gs in strata.items():
        rng.shuffle(gs)
        n = len(gs)
        n_te = int(round(n * fte))
        n_va = int(round(n * fva))
        if n >= 3 and fte > 0 and n_te == 0:
            n_te = 1
        if n >= 3 and n_va == 0:
            n_va = 1
        for g in gs[:n_te]:
            split["test"].extend(groups[g])
        for g in gs[n_te:n_te + n_va]:
            split["val"].extend(groups[g])
        for g in gs[n_te + n_va:]:
            split["train"].extend(groups[g])
    return split


def check_no_leakage(split):
    gs = {k: {it["group"] for it in v} for k, v in split.items()}
    inter = (gs["train"] & gs["val"]) | (gs["train"] & gs["test"]) | (gs["val"] & gs["test"])
    assert not inter, f"Vazamento entre partições: {list(inter)[:5]}"
    return True


def materialize(split, out_dir, class_names):
    out = Path(out_dir)
    if out.exists():
        shutil.rmtree(out)
    for part, its in split.items():
        (out / part / "images").mkdir(parents=True, exist_ok=True)
        (out / part / "labels").mkdir(parents=True, exist_ok=True)
        for k, it in enumerate(its):
            name = f"{k:05d}_{Path(it['img']).stem[:60]}"
            shutil.copy2(it["img"], out / part / "images" / (name + Path(it["img"]).suffix.lower()))
            dst = out / part / "labels" / (name + ".txt")
            if it["lbl"]:
                shutil.copy2(it["lbl"], dst)
            else:
                dst.write_text("")
    data = {"path": str(out.resolve()), "train": "train/images", "val": "val/images",
            "test": "test/images", "names": {i: n for i, n in enumerate(class_names)}}
    (out / "data.yaml").write_text(yaml.safe_dump(data, allow_unicode=True, sort_keys=False))
    return str(out / "data.yaml")


def split_summary(split, class_names):
    rows = []
    for part, its in split.items():
        inst = [0] * len(class_names)
        img_with = [0] * len(class_names)
        for it in its:
            for c in it["classes"]:
                if c < len(inst):
                    inst[c] += 1
            for c in set(it["classes"]):
                if c < len(img_with):
                    img_with[c] += 1
        row = {"particao": part, "imagens": len(its), "grupos": len({i["group"] for i in its})}
        for i, n in enumerate(class_names):
            row[f"inst_{n}"] = inst[i]
            row[f"imgs_{n}"] = img_with[i]
        rows.append(row)
    return rows


# ----------------------------------------------------------------------------
# 3. Treino e avaliação
# ----------------------------------------------------------------------------
def train_one(data_yaml, model_name, seed, epochs, imgsz, batch, patience, project, name, **hyp):
    from ultralytics import YOLO
    model = YOLO(model_name)
    model.train(data=data_yaml, epochs=epochs, imgsz=imgsz, batch=batch, patience=patience,
                seed=seed, deterministic=True, project=project, name=name, exist_ok=True,
                plots=True, verbose=False, **hyp)
    save_dir = Path(model.trainer.save_dir)   # robusto a mudanças de layout entre versões
    return str(save_dir / "weights" / "best.pt"), str(save_dir)


def evaluate(weights, data_yaml, split, imgsz, project, name, conf=0.001, iou=0.6):
    """Avalia em `split` ('val' ou 'test'); retorna métricas globais e por classe."""
    from ultralytics import YOLO
    m = YOLO(weights)
    r = m.val(data=data_yaml, split=split, imgsz=imgsz, conf=conf, iou=iou, plots=True,
              project=project, name=name, exist_ok=True, verbose=False)
    names = r.names
    out = {"global": {}, "per_class": {}, "save_dir": str(r.save_dir)}
    for kind, met in (("box", r.box), ("mask", getattr(r, "seg", None))):
        if met is None:
            continue
        out["global"][kind] = {"P": float(met.mp), "R": float(met.mr),
                               "mAP50": float(met.map50), "mAP50_95": float(met.map)}
        for j, c in enumerate(met.ap_class_index):
            cname = names[int(c)]
            out["per_class"].setdefault(cname, {})[kind] = {
                "P": float(met.p[j]), "R": float(met.r[j]),
                "AP50": float(met.ap50[j]), "AP50_95": float(met.ap[j])}
    cm = getattr(r, "confusion_matrix", None)
    if cm is not None:
        out["confusion_matrix"] = cm.matrix.astype(int).tolist()
        out["confusion_labels"] = [names[i] for i in range(len(names))] + ["background"]
    return out


def _iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def false_negative_audit(weights, data_yaml, class_id, split="test", imgsz=640,
                         thresholds=(0.10, 0.25, 0.40, 0.50), iou_thr=0.5):
    """Para a classe `class_id` (ex.: triatomíneo), conta acertos e falsos
    negativos em diferentes limiares de confiança, e lista imagens com perdas.
    Usa caixas envolventes dos polígonos (suficiente para triagem)."""
    from ultralytics import YOLO
    cfg = yaml.safe_load(Path(data_yaml).read_text())
    img_dir = Path(cfg["path"]) / cfg[split]
    lbl_dir = img_dir.parent / "labels"
    m = YOLO(weights)
    res = {t: {"TP": 0, "FN": 0, "FP": 0, "missed": []} for t in thresholds}
    imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT)
    for p in imgs:
        gts = []
        lf = lbl_dir / (p.stem + ".txt")
        if lf.exists():
            for line in lf.read_text().splitlines():
                v = line.split()
                if v and int(float(v[0])) == class_id:
                    xy = np.array(v[1:], dtype=float).reshape(-1, 2)
                    if xy.shape[0] == 2 and len(v) == 5:  # caixa xywh
                        cx, cy, w, h = map(float, v[1:5])
                        gts.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
                    else:
                        gts.append([xy[:, 0].min(), xy[:, 1].min(), xy[:, 0].max(), xy[:, 1].max()])
        pred = m.predict(str(p), imgsz=imgsz, conf=min(thresholds), verbose=False)[0]
        boxes = pred.boxes
        pb = boxes.xyxyn.cpu().numpy() if boxes is not None else np.zeros((0, 4))
        pc = boxes.cls.cpu().numpy().astype(int) if boxes is not None else np.zeros(0, int)
        ps = boxes.conf.cpu().numpy() if boxes is not None else np.zeros(0)
        for t in thresholds:
            cand = [b for b, c, s in zip(pb, pc, ps) if c == class_id and s >= t]
            used = set()
            for g in gts:
                hit = None
                for k, b in enumerate(cand):
                    if k not in used and _iou_xyxy(g, b) >= iou_thr:
                        hit = k
                        break
                if hit is None:
                    res[t]["FN"] += 1
                    res[t]["missed"].append(p.name)
                else:
                    used.add(hit)
                    res[t]["TP"] += 1
            res[t]["FP"] += len(cand) - len(used)
    for t in thresholds:
        tp, fn, fp = res[t]["TP"], res[t]["FN"], res[t]["FP"]
        res[t]["recall"] = tp / (tp + fn) if (tp + fn) else float("nan")
        res[t]["precision"] = tp / (tp + fp) if (tp + fp) else float("nan")
        res[t]["missed"] = sorted(set(res[t]["missed"]))
    return res


# ----------------------------------------------------------------------------
# 4. Estatísticas entre sementes
# ----------------------------------------------------------------------------
_T975 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365,
         8: 2.306, 9: 2.262, 10: 2.228}


def mean_ci(values):
    v = np.array([x for x in values if x is not None and not math.isnan(x)], dtype=float)
    n = len(v)
    if n == 0:
        return {"n": 0}
    mean = float(v.mean())
    sd = float(v.std(ddof=1)) if n > 1 else 0.0
    half = _T975.get(n - 1, 1.96) * sd / math.sqrt(n) if n > 1 else float("nan")
    return {"n": n, "mean": mean, "sd": sd, "ci95_low": mean - half, "ci95_high": mean + half}


def flatten_metrics(ev):
    flat = {}
    for kind, d in ev["global"].items():
        for k, v in d.items():
            flat[f"all/{kind}/{k}"] = v
    for cname, kinds in ev["per_class"].items():
        for kind, d in kinds.items():
            for k, v in d.items():
                flat[f"{cname}/{kind}/{k}"] = v
    return flat


def aggregate(runs):
    keys = sorted({k for r in runs for k in r})
    return {k: mean_ci([r.get(k) for r in runs]) for k in keys}


def environment_info(extra=None):
    info = {"python": platform.python_version(), "platform": platform.platform()}
    try:
        import torch
        import ultralytics
        info.update({"ultralytics": ultralytics.__version__, "torch": torch.__version__,
                     "cuda": torch.version.cuda,
                     "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})
    except Exception as e:  # pragma: no cover
        info["error"] = str(e)
    if extra:
        info.update(extra)
    return info


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUT = Path('/content/drive/MyDrive/artigo_acai/resultados_yolo')   # resultados ficam salvos no Drive
OUT.mkdir(parents=True, exist_ok=True)
DATA_DIRS, SOURCE_TAGS = [], []
print(OUT)

## 2. Dados

**Importante:** exportem do Roboflow uma versão **sem aumento de dados** (Generate → Augmentation: *none*), só com auto-orient e resize.
O aumento passa a ser feito apenas no treino, pelo próprio Ultralytics. Se só houver a versão aumentada, a Seção 4 remove as cópias automaticamente (`DEDUPE=True`).

Fontes possíveis (adicionem quantas tiverem — cada uma vira uma `SOURCE_TAG`):
1. **Projeto Roboflow de vocês** (opção A abaixo).
2. **Fotos novas de campo/batedores**, com GPS e data no EXIF (celular), anotadas no Roboflow — ideal para o *teste externo*.
3. **Triatomíneos — Virtual Vector Lab** (Gurgel-Gonçalves et al., 2017; Dryad doi:10.5061/dryad.br14k): ~1.900 fotos de 67 espécies brasileiras em vista dorsal. Fundo azul padronizado, portanto ajuda no treino mas **não** substitui fotos em contexto real; precisam ser anotadas (polígono) antes.
4. **iNaturalist** (filtro: *Triatominae*, Brasil, grau de pesquisa, licenças CC-BY/CC0): fotos em contexto real, georreferenciadas.


In [ ]:
# ---- Opção A: baixar do Roboflow (coloquem a chave em Segredos do Colab: ROBOFLOW_API_KEY)
USE_ROBOFLOW = False   # True para baixar direto do Roboflow; False para usar o zip final (opção B)
RF_WORKSPACE = "SEU-WORKSPACE"      # <- editar
RF_PROJECT   = "SEU-PROJETO"        # <- editar
RF_VERSIONS  = [2]                  # <- versões a baixar (ex.: [1, 2])

if USE_ROBOFLOW:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    proj = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    for v in RF_VERSIONS:
        ds = proj.version(v).download("yolov11", location=f"/content/datasets/rf_v{v}", overwrite=True)
        DATA_DIRS.append(ds.location); SOURCE_TAGS.append(f"roboflow_v{v}")
print(DATA_DIRS)

In [ ]:
# ---- Opção B: zips no Drive (formato YOLO segmentação, exportado do Roboflow)
import zipfile, glob
EXTRA_ZIPS = {
    "acai_artigo_final": "/content/drive/MyDrive/artigo_acai/acai_artigo_final_yolo.zip",   # zip final corrigido
    # "campo_2026": "/content/drive/MyDrive/artigo_acai_yolo/zips/campo_2026.zip",
    # "vvl_triatomineos": "/content/drive/MyDrive/artigo_acai_yolo/zips/vvl.zip",
}
for tag, z in EXTRA_ZIPS.items():
    dst = f"/content/datasets/{tag}"
    zipfile.ZipFile(z).extractall(dst)
    DATA_DIRS.append(dst); SOURCE_TAGS.append(tag)
print(list(zip(SOURCE_TAGS, DATA_DIRS)))

## 3. Configuração do experimento

In [ ]:
CLASS_NAMES = ["acai", "triatomineo"]   # mesma ordem do data.yaml (a-ai = 0, barbeiro = 1)
TRIATOMINE_ID = 1                        # índice da classe triatomíneo (None no experimento só-açaí)

MODELS   = ["yolo11s-seg.pt"]            # pode comparar: ["yolo11n-seg.pt","yolo11s-seg.pt","yolo11m-seg.pt"]
SEEDS    = [0, 1, 2, 3, 4]               # 5 repetições
EPOCHS   = 300
PATIENCE = 50                            # early stopping
IMGSZ    = 640
BATCH    = 16
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)
SPLIT_SEED = 42                          # a divisão é fixa; as sementes variam só o treino
HOLDOUT_SOURCES = []                     # ex.: ["campo_2026"] -> teste externo independente
DEDUPE = True                            # remove cópias aumentadas do Roboflow

HYP = dict(fliplr=0.5, degrees=15, hsv_v=0.15, mosaic=1.0, mixup=0.0,
           close_mosaic=10, optimizer="auto", cos_lr=True)
SELECTION_CRITERION = "best.pt = maior fitness na validação (0.1*mAP50 + 0.9*mAP50-95, caixa+máscara; padrão Ultralytics)"


## 4. Divisão por grupo (sem vazamento)

In [ ]:
import pandas as pd
import acai_yolo_pipeline as P

items = P.collect_items(DATA_DIRS, SOURCE_TAGS)
print("imagens encontradas:", len(items))
if DEDUPE:
    items = P.dedupe_augmented(items)
    print("após remover variantes aumentadas:", len(items))

split = P.group_split(items, SPLIT_FRACTIONS, seed=SPLIT_SEED, holdout_sources=HOLDOUT_SOURCES)
P.check_no_leakage(split)
DATA_YAML = P.materialize(split, "/content/ds_split", CLASS_NAMES)

summary = pd.DataFrame(P.split_summary(split, CLASS_NAMES))
(OUT/"tabelas").mkdir(exist_ok=True); summary.to_csv(OUT/"tabelas"/"divisao_dataset.csv", index=False)
summary

## 5. Treino com várias sementes + avaliação em validação e TESTE

In [ ]:
import json
runs = []
for model_name in MODELS:
    for seed in SEEDS:
        name = f"{Path(model_name).stem}_seed{seed}"
        print("=== treinando", name)
        best, train_dir = P.train_one(DATA_YAML, model_name, seed, EPOCHS, IMGSZ, BATCH, PATIENCE,
                                      project=str(OUT/"runs"), name=name, **HYP)
        ev_val  = P.evaluate(best, DATA_YAML, "val",  IMGSZ, str(OUT/"runs"), name + "_val")
        ev_test = P.evaluate(best, DATA_YAML, "test", IMGSZ, str(OUT/"runs"), name + "_test")
        fn = P.false_negative_audit(best, DATA_YAML, TRIATOMINE_ID, "test", IMGSZ) if TRIATOMINE_ID is not None else None
        res_csv = pd.read_csv(Path(train_dir)/"results.csv"); res_csv.columns = res_csv.columns.str.strip()
        rec = {"model": model_name, "seed": seed, "weights": best, "train_dir": train_dir,
               "epochs_run": int(res_csv["epoch"].max()),
               "best_epoch": int(res_csv.loc[res_csv.filter(like="metrics/mAP50-95").sum(axis=1).idxmax(), "epoch"]),
               "val": ev_val, "test": ev_test, "fn_audit_test": fn}
        runs.append(rec)
        P.save_json(rec, Path(train_dir)/"avaliacao.json")
        print(json.dumps(ev_test["global"], indent=1))

## 6. Agregação entre sementes (média ± DP, IC 95%)

In [ ]:
def table(runs, part):
    rows = []
    for model_name in MODELS:
        rs = [P.flatten_metrics(r[part]) for r in runs if r["model"] == model_name]
        agg = P.aggregate(rs)
        for k, s in agg.items():
            if s.get("n"):
                scope, kind, metric = k.split("/")
                rows.append({"modelo": Path(model_name).stem, "particao": part, "classe": scope,
                             "tipo": kind, "metrica": metric, "n_sementes": s["n"],
                             "media": round(s["mean"], 3), "dp": round(s["sd"], 3),
                             "ic95_inf": round(s["ci95_low"], 3), "ic95_sup": round(s["ci95_high"], 3)})
    return pd.DataFrame(rows)

agg = pd.concat([table(runs, "val"), table(runs, "test")])
agg.to_csv(OUT/"tabelas"/"metricas_agregadas.csv", index=False)

# tabela compacta para o artigo: teste, máscara, por classe
art = agg[(agg.particao=="test") & (agg.tipo=="mask")].copy()
art["valor"] = art.apply(lambda r: f"{r.media:.3f} ± {r.dp:.3f} [{r.ic95_inf:.3f}; {r.ic95_sup:.3f}]", axis=1)
art = art.pivot_table(index=["modelo","classe"], columns="metrica", values="valor", aggfunc="first")
art.to_csv(OUT/"tabelas"/"tabela_artigo_teste_mascara.csv"); art

In [ ]:
# Falsos negativos de triatomíneos no TESTE, por limiar de confiança (média entre sementes)
if TRIATOMINE_ID is not None:
    fr = []
    for r in runs:
        for t, d in r["fn_audit_test"].items():
            fr.append({"modelo": Path(r["model"]).stem, "seed": r["seed"], "limiar": float(t),
                       "TP": d["TP"], "FN": d["FN"], "FP": d["FP"],
                       "revocacao": d["recall"], "precisao": d["precision"], "imagens_perdidas": "; ".join(d["missed"])})
    fr = pd.DataFrame(fr); fr.to_csv(OUT/"tabelas"/"falsos_negativos_triatomineos.csv", index=False)
    display(fr.groupby(["modelo","limiar"])[["TP","FN","FP","revocacao","precisao"]].agg(["mean","std"]).round(3))

In [ ]:
# Épocas executadas e melhor época por semente (para o texto de Métodos/Resultados)
ep = pd.DataFrame([{k: r[k] for k in ("model","seed","epochs_run","best_epoch")} for r in runs])
ep.to_csv(OUT/"tabelas"/"epocas.csv", index=False); ep

## 7. Figuras legíveis para o artigo (parecer: Figuras 5–8)

In [ ]:
import matplotlib.pyplot as plt, shutil
plt.rcParams.update({"font.size": 13, "axes.titlesize": 14, "axes.labelsize": 13, "legend.fontsize": 11})
FIG = OUT/"figuras"; FIG.mkdir(exist_ok=True)

def plot_curves(run_dir, title, fname):
    df = pd.read_csv(Path(run_dir)/"results.csv"); df.columns = df.columns.str.strip()
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    for col, lab in [("train/seg_loss","treino"), ("val/seg_loss","validação")]:
        if col in df: ax[0].plot(df.epoch, df[col], label=lab, lw=2)
    ax[0].set(title="Perda de segmentação", xlabel="Época", ylabel="Perda"); ax[0].legend()
    for col, lab in [("metrics/precision(M)","Precisão"), ("metrics/recall(M)","Revocação")]:
        if col in df: ax[1].plot(df.epoch, df[col], label=lab, lw=2)
    ax[1].set(title="Validação (máscara)", xlabel="Época", ylim=(0,1)); ax[1].legend()
    for col, lab in [("metrics/mAP50(M)","mAP50"), ("metrics/mAP50-95(M)","mAP50–95")]:
        if col in df: ax[2].plot(df.epoch, df[col], label=lab, lw=2)
    ax[2].set(title="Validação (máscara)", xlabel="Época", ylim=(0,1)); ax[2].legend()
    fig.suptitle(title); fig.tight_layout(); fig.savefig(FIG/fname, dpi=300); plt.show()

# semente com mAP50-95 de máscara no teste mais próximo da mediana (figura representativa, não a melhor)
vals = [(r["test"]["global"]["mask"]["mAP50_95"], r) for r in runs]
vals.sort(key=lambda x: x[0]); rep = vals[len(vals)//2][1]
name = f"{Path(rep['model']).stem}_seed{rep['seed']}"
plot_curves(rep["train_dir"], f"Curvas de treino — {name} (semente representativa)", "curvas_treino.png")
for f in ["confusion_matrix_normalized.png", "MaskPR_curve.png", "BoxPR_curve.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    src = Path(rep["test"]["save_dir"])/f
    if src.exists(): shutil.copy(src, FIG/("teste_" + f))
print(sorted(p.name for p in FIG.iterdir()))

## 8. Metadados de reprodutibilidade

In [ ]:
meta = P.environment_info({
    "models_initial_weights": MODELS, "seeds": SEEDS, "epochs_max": EPOCHS, "patience": PATIENCE,
    "imgsz": IMGSZ, "batch": BATCH, "hyperparameters": HYP, "split_fractions": SPLIT_FRACTIONS,
    "split_seed": SPLIT_SEED, "holdout_sources": HOLDOUT_SOURCES, "dedupe_augmented": DEDUPE,
    "sources": dict(zip(SOURCE_TAGS, DATA_DIRS)), "class_names": CLASS_NAMES,
    "selection_criterion": SELECTION_CRITERION, "eval_conf": 0.001, "eval_iou": 0.6,
    "fn_audit_iou": 0.5})
P.save_json(meta, OUT/"run_metadata.json"); meta

In [ ]:
!cd "{OUT}" && zip -qr resultados_yolo_acai.zip tabelas figuras run_metadata.json && find runs -name "avaliacao.json" -o -name "results.csv" | zip -q resultados_yolo_acai.zip -@
print("Enviem este arquivo:", OUT/"resultados_yolo_acai.zip")

## (Opcional) Validação cruzada por grupo (5 folds)
Com bases pequenas (< 300 imagens), a validação cruzada por grupo dá estimativas mais estáveis que uma única divisão. Descomente para rodar (≈ 5× o tempo).

In [ ]:
# from sklearn.model_selection import StratifiedGroupKFold
# import numpy as np
# groups = [it["group"] for it in items]
# strat  = [int(TRIATOMINE_ID in it["classes"]) if TRIATOMINE_ID is not None else 0 for it in items]
# cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
# cv_runs = []
# for k, (tr, te) in enumerate(cv.split(items, strat, groups)):
#     tr_items = [items[i] for i in tr]
#     inner = P.group_split(tr_items, (0.85, 0.15, 0.0), seed=k)
#     sp = {"train": inner["train"], "val": inner["val"], "test": [items[i] for i in te]}
#     P.check_no_leakage(sp)
#     y = P.materialize(sp, f"/content/cv_fold{k}", CLASS_NAMES)
#     best, _ = P.train_one(y, MODELS[0], 0, EPOCHS, IMGSZ, BATCH, PATIENCE, str(OUT/"cv"), f"fold{k}", **HYP)
#     cv_runs.append(P.flatten_metrics(P.evaluate(best, y, "test", IMGSZ, str(OUT/"cv"), f"fold{k}_test")))
# pd.DataFrame(P.aggregate(cv_runs)).T.round(3)